In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from PIL import Image

# CONFIG
DATA_DIR = r"C:\Computer Vision Project\archive (2)\lung_colon_image_set\Train and Validation Set"
IMG_SIZE = 224
BATCH_SIZE = 16
SEED = 42
FOLDS = 5
EPOCHS = 15

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

TARGET_CLASSES = ['lung_aca', 'lung_n', 'lung_scc']

Device: cuda


In [2]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [3]:
class FilteredImageFolder(Dataset):
    def __init__(self, root_dir, classes, transform=None):
        self.dataset = datasets.ImageFolder(root=root_dir)
        self.transform = transform

        self.class_to_idx = {cls: i for i, cls in enumerate(classes)}
        self.samples = []

        for path, label in self.dataset.samples:
            class_name = self.dataset.classes[label]
            if class_name in classes:
                self.samples.append((path, self.class_to_idx[class_name]))

        self.classes = classes

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [4]:
full_dataset = FilteredImageFolder(
    root_dir=DATA_DIR,
    classes=TARGET_CLASSES,
    transform=None  # Important for CV
)

# Extract labels for stratification
labels = [label for _, label in full_dataset.samples]

print("Total samples:", len(labels))

Total samples: 13501


In [5]:
def get_model():
    weights = EfficientNet_B0_Weights.DEFAULT
    model = efficientnet_b0(weights=weights)

    # Partial unfreeze
    for param in model.features[:5].parameters():
        param.requires_grad = False
    for param in model.features[5:].parameters():
        param.requires_grad = True

    in_features = model.classifier[1].in_features

    model.classifier = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, 3)
    )

    return model.to(device)

In [6]:
def train_one_fold(model, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=3e-4,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=2, factor=0.3
    )

    best_acc = 0

    for epoch in range(EPOCHS):
        model.train()
        train_preds, train_labels = [], []

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            preds = torch.argmax(outputs, 1)
            train_preds.extend(preds.cpu().numpy())
            train_labels.extend(labels.cpu().numpy())

        train_acc = accuracy_score(train_labels, train_preds)

        # Validation
        model.eval()
        val_preds, val_labels = [], []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                outputs = model(images)
                preds = torch.argmax(outputs, 1)

                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.numpy())

        val_acc = accuracy_score(val_labels, val_preds)
        scheduler.step(val_acc)

        print(f"Epoch {epoch+1}: Train={train_acc:.4f}, Val={val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc

    return best_acc

In [7]:
skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

fold_accuracies = []

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    print(f"\n===== FOLD {fold+1} =====")

    # Subsets
    train_subset = Subset(full_dataset, train_idx)
    val_subset   = Subset(full_dataset, val_idx)

    # Assign transforms
    train_subset.dataset.transform = train_transform
    val_subset.dataset.transform   = val_transform

    # Loaders
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader   = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # Model
    model = get_model()

    # Train
    best_acc = train_one_fold(model, train_loader, val_loader)
    fold_accuracies.append(best_acc)

    print(f"Fold {fold+1} Best Accuracy: {best_acc:.4f}")


===== FOLD 1 =====
Epoch 1: Train=0.9622, Val=0.9967
Epoch 2: Train=0.9911, Val=1.0000
Epoch 3: Train=0.9946, Val=1.0000
Epoch 4: Train=0.9975, Val=0.9967
Epoch 5: Train=0.9966, Val=1.0000
Epoch 6: Train=0.9994, Val=1.0000
Epoch 7: Train=0.9996, Val=1.0000
Epoch 8: Train=0.9997, Val=1.0000
Epoch 9: Train=0.9999, Val=0.9996
Epoch 10: Train=0.9997, Val=1.0000
Epoch 11: Train=0.9998, Val=1.0000
Epoch 12: Train=0.9998, Val=1.0000
Epoch 13: Train=0.9998, Val=1.0000
Epoch 14: Train=0.9999, Val=1.0000
Epoch 15: Train=1.0000, Val=1.0000
Fold 1 Best Accuracy: 1.0000

===== FOLD 2 =====
Epoch 1: Train=0.9613, Val=0.9967
Epoch 2: Train=0.9920, Val=0.9952
Epoch 3: Train=0.9978, Val=0.9944
Epoch 4: Train=0.9956, Val=0.9985
Epoch 5: Train=0.9949, Val=0.9978
Epoch 6: Train=0.9972, Val=0.9996
Epoch 7: Train=0.9970, Val=0.9996
Epoch 8: Train=0.9988, Val=0.9978
Epoch 9: Train=0.9956, Val=0.9993
Epoch 10: Train=0.9993, Val=0.9989
Epoch 11: Train=0.9998, Val=0.9993
Epoch 12: Train=0.9993, Val=0.9985
Epoc

In [9]:
print("\n===== CROSS VALIDATION RESULTS =====")
for i, acc in enumerate(fold_accuracies):
    print(f"Fold {i+1}: {acc:.4f}")

print(f"\nMean Accuracy: {np.mean(fold_accuracies):.4f}")
print(f"Std Dev: {np.std(fold_accuracies):.4f}")


===== CROSS VALIDATION RESULTS =====
Fold 1: 1.0000
Fold 2: 0.9996
Fold 3: 1.0000
Fold 4: 1.0000
Fold 5: 1.0000

Mean Accuracy: 0.9999
Std Dev: 0.0001
